# CASE: PERFORMANCE DE PAGAMENTOS - iFOOD PAGO
## Análise de performance transacional por adquirente
============================================================

## Estrutura:
##### 0. Setup e ingestao dos dados (Google Sheets)
##### 1. Q1 - Entendimento dos dados (SQL de juncao)
##### 2. Q2 - Analise de TPV ao longo do periodo
##### 3. Q3 - Conversao transacional
##### 4. Q4 - TPV recusado e o que representa para o negocio
##### 5. Q5 - Performance dos parceiros (multidimensional)
##### 6. Q6 - Oportunidades de melhoria e priorizacao
 ============================================================


## Fase 0 — Setup e Ingestão dos Dados

Importação das 3 bases a partir de um Google Sheets compartilhado e criação de tabelas temporárias para uso em SQL.

In [0]:
# ============================================================
# 0.1 — Importações
# ============================================================
import pandas as pd
import numpy as np

In [0]:
# ============================================================
# 0.2 — Leitura das 3 bases do Google Sheets
# ============================================================
# O Google Sheets está compartilhado como "qualquer pessoa com o link pode ver"
# Cada aba é exportada como CSV via URL de exportação (format=csv&gid=...)

# Aba 1: base_transacao — dados transacionais (valor, status, código de erro, método, bandeira, data)
url_transacao = "https://docs.google.com/spreadsheets/d/1UcHc42kNlnBwfinj_ymHv_aAP6zXIdTjELnMhGwkeHk/export?format=csv&gid=0"

# Aba 2: base_adquirente — mapeia transaction_id para cada adquirente (A, B, C, D)
url_adquirente = "https://docs.google.com/spreadsheets/d/1UcHc42kNlnBwfinj_ymHv_aAP6zXIdTjELnMhGwkeHk/export?format=csv&gid=667212389"

# Aba 3: base_negocio_ifood — mapeia transaction_id para a linha de negócio (Delivery, Market, Club)
url_negocio = "https://docs.google.com/spreadsheets/d/1UcHc42kNlnBwfinj_ymHv_aAP6zXIdTjELnMhGwkeHk/export?format=csv&gid=1422199832"

# Leitura como pandas DataFrame
df_transacao = pd.read_csv(url_transacao)
df_adquirente = pd.read_csv(url_adquirente)
df_negocio = pd.read_csv(url_negocio)

print(f"base_transacao:    {df_transacao.shape[0]} linhas x {df_transacao.shape[1]} colunas")
print(f"base_adquirente:    {df_adquirente.shape[0]} linhas x {df_adquirente.shape[1]} colunas")
print(f"base_negocio_ifood: {df_negocio.shape[0]} linhas x {df_negocio.shape[1]} colunas")

base_transacao:    1000 linhas x 9 colunas
base_adquirente:    931 linhas x 2 colunas
base_negocio_ifood: 1000 linhas x 2 colunas


In [0]:
# ============================================================
# 0.3 — Tratamento básico
# ============================================================
# card_issuer_code vem como string 'null' em transações PIX — converter para nulo real
df_transacao['card_issuer_code'] = df_transacao['card_issuer_code'].replace('null', np.nan)

# amount pode vir como string com vírgula decimal — forçar para float
if df_transacao['amount'].dtype == 'object':
    df_transacao['amount'] = df_transacao['amount'].astype(str).str.replace(',', '.').astype(float)

# Garantir que transaction_dt está em formato de data
df_transacao['transaction_dt'] = pd.to_datetime(df_transacao['transaction_dt'])

# Verificação
print(f"amount: min R$ {df_transacao['amount'].min():.2f} | max R$ {df_transacao['amount'].max():.2f} | tipo: {df_transacao['amount'].dtype}")
print(f"dt: {df_transacao['transaction_dt'].dtype}")

# Recriar a temp view com os dados tratados
spark.createDataFrame(df_transacao).createOrReplaceTempView("base_transacao")

print("base_transacao recriada com amount como float.")

amount: min R$ 0.01 | max R$ 440.27 | tipo: float64
dt: datetime64[ns]
base_transacao recriada com amount como float.


## Fase 1 — Entendimento dos Dados (Q1)

Propor a lógica de uma consulta SQL que organiza as 3 bases em uma tabela analítica unificada.
O objetivo é deixar claros os relacionamentos, as decisões de modelagem e as informações importantes para a análise.

In [0]:
%sql
-- ============================================================
--1.1 — CONSULTA SQL DE JUNÇÃO DAS 3 BASES
-- ============================================================
-- A consulta abaixo cria uma view temporária unificada que serve
-- de base para todas as análises Q2-Q6.

CREATE OR REPLACE TEMP VIEW vw_transacoes_unificadas AS
SELECT 
    t.payment_method,
    t.card_issuer_code,
    t.card_brand,
    t.amount,
    t.transaction_id,
    CAST(t.transaction_dt AS DATE) AS transaction_date,
    t.transaction_event,
    t.transaction_status,
    t.return_code,
    
    -- Adquirente (LEFT JOIN preserva todas as transações)
    COALESCE(a.adquirente, 'Não identificado') AS adquirente,
    
    -- Negócio iFood (backtick no nome da coluna com acento)
    n.`operação` AS negocio_ifood,
    
    -- Flags de análise
    CASE WHEN t.transaction_status = 'SUCCEEDED' THEN 1 ELSE 0 END AS flag_sucesso,
    CASE WHEN t.transaction_status = 'DECLINED'  THEN 1 ELSE 0 END AS flag_recusa,
    
    -- Classificação do código de erro por responsável
    CASE 
        WHEN t.return_code IS NULL THEN 'Sem recusa'
        WHEN t.return_code = 51    THEN 'Cliente/Emissor - Saldo insuficiente'
        WHEN t.return_code = 5     THEN 'Emissor - Do not honor (ambíguo)'
        WHEN t.return_code = 91    THEN 'Infraestrutura - Emissor fora do ar'
        ELSE 'Código não mapeado'
    END AS descricao_erro,
    
    -- Classificação por reversibilidade
    CASE 
        WHEN t.return_code IS NULL          THEN NULL
        WHEN t.return_code IN (51, 91)      THEN 'Recuperável'
        WHEN t.return_code = 5              THEN 'Ambíguo/Não recuperável'
        ELSE 'Não classificado'
    END AS recuperabilidade
    
FROM base_transacao t
LEFT JOIN base_adquirente a 
    ON t.transaction_id = a.transaction_id
LEFT JOIN base_negocio_ifood n 
    ON t.transaction_id = n.transaction_id
;
SELECT * FROM vw_transacoes_unificadas

payment_method,card_issuer_code,card_brand,amount,transaction_id,transaction_date,transaction_event,transaction_status,return_code,adquirente,negocio_ifood,flag_sucesso,flag_recusa,descricao_erro,recuperabilidade
CREDIT,null,MASTERCARD,13.9,ffd021d9-bb84-4cac-af0b-28fc1989f436,2024-01-29,CAPTURE,SUCCEEDED,null,Adquirente D,IFOOD_BR_DELIVERY,1,0,Sem recusa,null
CREDIT,341.0,MASTERCARD,24.99,ffcafbd6-416b-4635-8074-a55e03f9f2ba,2024-01-29,CAPTURE,SUCCEEDED,null,Adquirente D,IFOOD_BR_DELIVERY,1,0,Sem recusa,null
CREDIT,341.0,VISA,58.4,ffc6dac9-cd82-4e46-b373-a89f1f3171f8,2024-01-29,CAPTURE,SUCCEEDED,null,Adquirente D,IFOOD_BR_DELIVERY,1,0,Sem recusa,null
CREDIT,237.0,MASTERCARD,4.95,ff814ea4-0df0-4d97-9bb6-29e72b4af95f,2024-01-29,CAPTURE,SUCCEEDED,null,Não identificado,IFOOD_BR_CLUB,1,0,Sem recusa,null
PIX,null,PIX,34.97,ff7b54cf-19e8-41d4-9588-1c0e5e7a3f9f,2024-01-29,CAPTURE,SUCCEEDED,null,Adquirente D,IFOOD_BR_DELIVERY,1,0,Sem recusa,null
CREDIT,623.0,MASTERCARD,34.9,ff3c153f-c3c4-4c67-801c-d1f84912b06f,2024-01-29,CAPTURE,SUCCEEDED,null,Adquirente D,IFOOD_BR_DELIVERY,1,0,Sem recusa,null
CREDIT,33.0,VISA,128.99,fefa1edb-8ab9-4e8a-a433-2bc09d21d69f,2024-01-29,CAPTURE,DECLINED,91.0,Adquirente D,IFOOD_BR_DELIVERY,0,1,Infraestrutura - Emissor fora do ar,Recuperável
PIX,null,PIX,76.63,fef52d3a-7169-40aa-8f38-4b74f8b95635,2024-01-29,CAPTURE,SUCCEEDED,null,Adquirente D,IFOOD_BR_MARKET,1,0,Sem recusa,null
PIX,null,PIX,5.95,feca68be-24ff-4aab-ba41-06fe392cadef,2024-01-29,CAPTURE,SUCCEEDED,null,Adquirente D,IFOOD_BR_DELIVERY,1,0,Sem recusa,null
CREDIT,102.0,VISA,40.49,fe68b3c6-d80b-44b0-93ba-d628dacb39a8,2024-01-29,CAPTURE,SUCCEEDED,null,Adquirente D,IFOOD_BR_DELIVERY,1,0,Sem recusa,null


In [0]:
%sql
-- ============================================================
-- 1.2 — VALIDAÇÃO DA JUNÇÃO
-- ============================================================
-- Verifica se a junção funcionou corretamente:
-- - Total de transações deve ser 1.000 (mesmo da base original)
-- - Sucessos + Recusas deve bater com o total
-- - Transações sem adquirente mapeado deve ser 69
-- ============================================================

SELECT 
    COUNT(*) AS total_transacoes,
    SUM(flag_sucesso) AS sucessos,
    SUM(flag_recusa) AS recusas,
    SUM(CASE WHEN adquirente = 'Não identificado' THEN 1 ELSE 0 END) AS sem_adquirente
FROM vw_transacoes_unificadas


total_transacoes,sucessos,recusas,sem_adquirente
1000,857,143,69


## Fase 2 — Análise de TPV ao longo do período (Q2)

Comportamentos, variações e pontos de atenção do Total Payment Volume (TPV) ao longo dos 6 dias disponíveis. Inclui decomposição do salto de volume entre dias e composição por adquirente.

In [0]:
%sql
-- ============================================================
-- 2.1 — TPV DIÁRIO: TOTAL, APROVADO, RECUSADO, TICKET MÉDIO
-- ============================================================
-- TPV = Total Payment Volume = soma do valor (amount) das transações
-- O TPV pode ser dividido em:
--   - TPV aprovado: transações com status SUCCEEDED
--   - TPV recusado: transações com status DECLINED
--

SELECT
    transaction_date,
    COUNT(*) AS transacoes,
    ROUND(SUM(amount), 2) AS tpv_total,
    ROUND(SUM(CASE WHEN flag_sucesso = 1 THEN amount END), 2) AS tpv_aprovado,
    ROUND(SUM(CASE WHEN flag_recusa = 1  THEN amount END), 2) AS tpv_recusado,
    ROUND(SUM(amount) / COUNT(*), 2) AS ticket_medio,
    ROUND(SUM(flag_sucesso) * 100.0 / COUNT(*), 1) AS conversao_pct,
    ROUND(SUM(CASE WHEN flag_recusa = 1 THEN amount END)
          * 100.0 / SUM(amount), 1) AS pct_tpv_recusado
FROM vw_transacoes_unificadas
GROUP BY transaction_date
ORDER BY transaction_date


transaction_date,transacoes,tpv_total,tpv_aprovado,tpv_recusado,ticket_medio,conversao_pct,pct_tpv_recusado
2024-01-22,49,1751.13,1584.72,166.41,35.74,89.8,9.5
2024-01-23,88,4372.2,4164.37,207.83,49.68,93.2,4.8
2024-01-24,88,4104.12,3723.33,380.79,46.64,89.8,9.3
2024-01-25,262,12885.52,10612.16,2273.36,49.18,82.4,17.6
2024-01-26,262,11629.57,9916.56,1713.01,44.39,83.2,14.7
2024-01-29,251,10982.48,9185.62,1796.86,43.75,86.9,16.4


In [0]:
%sql
-- ============================================================
-- 2.2 — COMPOSIÇÃO POR ADQUIRENTE POR DIA
-- ============================================================
-- Mostra qual adquirente processou em cada dia e sua conversão.
-- Achado-chave: as adquirentes operam em ROTAÇÃO (sequência de dias),
-- não em paralelo. Isso muda a leitura da "queda de performance".
-- ============================================================


SELECT
    transaction_date,
    adquirente,
    COUNT(*)                                          AS transacoes,
    ROUND(SUM(amount), 2)                              AS tpv,
    ROUND(SUM(flag_sucesso) * 100.0 / COUNT(*), 1)    AS conversao_pct
FROM vw_transacoes_unificadas
GROUP BY transaction_date, adquirente
ORDER BY transaction_date, adquirente


transaction_date,adquirente,transacoes,tpv,conversao_pct
2024-01-22,Adquirente A,46,1736.28,89.1
2024-01-22,Não identificado,3,14.85,100.0
2024-01-23,Adquirente A,81,4329.6,92.6
2024-01-23,Não identificado,7,42.6,100.0
2024-01-24,Adquirente A,84,4076.37,89.3
2024-01-24,Não identificado,4,27.75,100.0
2024-01-25,Adquirente A,189,9952.28,83.6
2024-01-25,Adquirente B,54,2815.34,75.9
2024-01-25,Não identificado,19,117.9,89.5
2024-01-26,Adquirente B,226,10663.96,84.5


In [0]:
%sql
-- 2.3 — EVIDÊNCIA DA ROTAÇÃO DE ADQUIRENTES
-- Pivot de dia x adquirente com o volume de transações
-- Os zeros mostram que cada adquirente opera em dias diferentes (rotação)

SELECT
    transaction_date,
    SUM(CASE WHEN adquirente = 'Adquirente A' THEN 1 ELSE 0 END) AS adq_a,
    SUM(CASE WHEN adquirente = 'Adquirente B' THEN 1 ELSE 0 END) AS adq_b,
    SUM(CASE WHEN adquirente = 'Adquirente C' THEN 1 ELSE 0 END) AS adq_c,
    SUM(CASE WHEN adquirente = 'Adquirente D' THEN 1 ELSE 0 END) AS adq_d,
    SUM(CASE WHEN adquirente = 'Não identificado' THEN 1 ELSE 0 END) AS nao_ident
FROM vw_transacoes_unificadas
GROUP BY transaction_date
ORDER BY transaction_date

transaction_date,adq_a,adq_b,adq_c,adq_d,nao_ident
2024-01-22,46,0,0,0,3
2024-01-23,81,0,0,0,7
2024-01-24,84,0,0,0,4
2024-01-25,189,54,0,0,19
2024-01-26,0,226,18,0,18
2024-01-29,0,0,152,81,18


In [0]:
%sql
-- 2.4 — DECOMPOSIÇÃO DO SALTO DE TPV (DIA 24 -> DIA 25)
-- O volume triplica do dia 24 para o dia 25 (88 -> 262 transações)
-- Efeito volume = (volume_novo x ticket_velho) - tpv_velho
-- Efeito ticket = tpv_novo - (volume_novo x ticket_velho)

WITH tpv_diario AS (
    SELECT
        transaction_date,
        COUNT(*) AS volume,
        SUM(amount) AS tpv,
        SUM(amount) / COUNT(*) AS ticket_medio
    FROM vw_transacoes_unificadas
    GROUP BY transaction_date
)
SELECT
    a.transaction_date AS dia_base,
    b.transaction_date AS dia_comparacao,
    a.volume AS vol_base,
    b.volume AS vol_comp,
    b.volume - a.volume AS delta_volume,
    ROUND(a.ticket_medio, 2) AS ticket_base,
    ROUND(b.ticket_medio, 2) AS ticket_comp,
    ROUND(b.tpv - a.tpv, 2) AS delta_tpv,
    ROUND((b.volume * a.ticket_medio) - a.tpv, 2) AS efeito_volume,
    ROUND(b.tpv - (b.volume * a.ticket_medio), 2) AS efeito_ticket
FROM tpv_diario a
CROSS JOIN tpv_diario b
WHERE a.transaction_date = '2024-01-24'
  AND b.transaction_date = '2024-01-25'

dia_base,dia_comparacao,vol_base,vol_comp,delta_volume,ticket_base,ticket_comp,delta_tpv,efeito_volume,efeito_ticket
2024-01-24,2024-01-25,88,262,174,46.64,49.18,8781.4,8114.96,666.44


### Conclusões Q2 — TPV ao longo do período

**Comportamento temporal:**
- O TPV total do período foi de **R$ 45.725,02**, dos quais **R$ 6.538,26 (14,3%)** foram recusados
- O volume de transações **triplica** do dia 24/01 (88 transações) para o dia 25/01 (262 transações)
- O ticket médio se mantém estável (~R$ 46), indicando que o salto de TPV vem de **volume**, não de ticket mais alto

**Decomposição do salto (dia 24 → 25):**
- **92% do salto de TPV** é explicado por efeito volume (mais transações)
- Apenas **8%** é explicado por mudança no ticket médio
- O salto coincide com a **entrada do Adquirente B** no processamento

**Rotação de adquirentes:**
- As adquirentes operam em **sequência** (rotação de dias), não em paralelo
- A: opera dias 22-25 | B: opera dias 25-26 | C: opera dias 26-29 | D: opera dia 29
- A "queda de performance" relatada no case pode estar parcialmente ligada a esse efeito de entrada de novas adquirentes

**Pontos de atenção:**
- A conversão cai de **93,2%** (dia 23) para **82,4%** (dia 25) — no mesmo dia em que o B entra
- O % TPV recusado sobe de **9,3%** (dia 24) para **17,6%** (dia 25)
- **Sábado e domingo não têm dados** — limitação da base que deve ser explicitada

**Premissa adotada:** por não haver dados anteriores ao período, a análise trabalha como um diagnóstico de snapshot (retrato de 6 dias), não como uma série histórica. A "queda" é interpretada como variação intra-período.

## Fase 3 — Conversão Transacional (Q3)

Cálculo da conversão geral e segmentada por adquirente, negócio e método de pagamento.
Fórmula: Conversão = Sucessos / Total

In [0]:
%sql
-- 3.1 — CONVERSÃO GERAL
-- Conversão = Sucessos / Total

SELECT
    COUNT(*) AS total,
    SUM(flag_sucesso) AS sucessos,
    SUM(flag_recusa) AS recusas,
    ROUND(SUM(flag_sucesso) * 100.0 / COUNT(*), 1) AS conversao_pct
FROM vw_transacoes_unificadas

total,sucessos,recusas,conversao_pct
1000,857,143,85.7


In [0]:
%sql
-- 3.2 — CONVERSÃO POR DIA (TRAJETÓRIA)
-- Mostra a evolução da conversão ao longo do período

SELECT
    transaction_date,
    COUNT(*) AS total,
    SUM(flag_sucesso) AS sucessos,
    ROUND(SUM(flag_sucesso) * 100.0 / COUNT(*), 1) AS conversao_pct
FROM vw_transacoes_unificadas
GROUP BY transaction_date
ORDER BY transaction_date

transaction_date,total,sucessos,conversao_pct
2024-01-22,49,44,89.8
2024-01-23,88,82,93.2
2024-01-24,88,79,89.8
2024-01-25,262,216,82.4
2024-01-26,262,218,83.2
2024-01-29,251,218,86.9


In [0]:
%sql
-- 3.3 — CONVERSÃO SEGMENTADA
-- Segmenta por adquirente, negócio e método em uma única consulta (UNION ALL)

SELECT 'Por adquirente' AS recorte, adquirente AS categoria,
       COUNT(*) AS total, SUM(flag_sucesso) AS sucessos,
       ROUND(SUM(flag_sucesso) * 100.0 / COUNT(*), 1) AS conv_pct
FROM vw_transacoes_unificadas
GROUP BY adquirente

UNION ALL

SELECT 'Por negocio' AS recorte, negocio_ifood AS categoria,
       COUNT(*) AS total, SUM(flag_sucesso) AS sucessos,
       ROUND(SUM(flag_sucesso) * 100.0 / COUNT(*), 1) AS conv_pct
FROM vw_transacoes_unificadas
GROUP BY negocio_ifood

UNION ALL

SELECT 'Por metodo' AS recorte, payment_method AS categoria,
       COUNT(*) AS total, SUM(flag_sucesso) AS sucessos,
       ROUND(SUM(flag_sucesso) * 100.0 / COUNT(*), 1) AS conv_pct
FROM vw_transacoes_unificadas
GROUP BY payment_method
ORDER BY recorte, conv_pct DESC

recorte,categoria,total,sucessos,conv_pct
Por adquirente,Não identificado,69,62,89.9
Por adquirente,Adquirente A,400,349,87.3
Por adquirente,Adquirente C,170,147,86.5
Por adquirente,Adquirente B,280,232,82.9
Por adquirente,Adquirente D,81,67,82.7
Por metodo,PIX,252,220,87.3
Por metodo,CREDIT,593,507,85.5
Por metodo,DEBIT,141,119,84.4
Por metodo,DIGITAL_WALLET,14,11,78.6
Por negocio,IFOOD_BR_CLUB,69,62,89.9


### Conclusões Q3 — Conversão transacional

**Resultado geral:**
- Conversão = 857 / 1.000 = **85,7%**
- 143 transações recusadas (14,3% do total)

**Trajetória:**
- A conversão cai de **93,2%** (dia 23/01) para **82,4%** (dia 25/01)
- A recuperação parcial no dia 29/01 (86,9%) não retorna ao nível inicial
- A queda coincide exatamente com a **entrada do Adquirente B** no processamento (dia 25)

**Segmentação:**
- **Por adquirente:** B (82,9%) e D (82,7%) têm as piores conversões; A (87,2%) e C (86,5%) as melhores
- **Por negócio:** IFOOD_BR_CLUB (89,9%) é o melhor; DELIVERY (85,1%) é o pior — mas a diferença é pequena
- **Por método:** DIGITAL_WALLET (78,6%) é o pior, mas com apenas 14 transações (amostra muito pequena); PIX (87,3%) é o melhor entre os métodos com volume relevante

**Premissa:** a conversão agregada de 85,7% esconde a diferença entre adquirentes. O número isolado não diz se é bom ou ruim — a segmentação é que mostra onde está o problema.

## Fase 4 — TPV Recusado e o que representa para o negócio (Q4)

Cálculo do TPV recusado total e por adquirente, com análise de recuperabilidade.
A pergunta central: quanto do TPV recusado tem chance de voltar (recuperável) e quanto está perdido (não recuperável)?

In [0]:
%sql
-- 4.1 — TPV RECUSADO GERAL
-- TPV recusado = soma do amount das transações com status DECLINED

SELECT
    COUNT(*) AS qtd_recusas,
    ROUND(SUM(amount), 2) AS tpv_recusado,
    ROUND(AVG(amount), 2) AS media,
    ROUND(APPROX_PERCENTILE(amount, 0.5), 2) AS mediana
FROM vw_transacoes_unificadas
WHERE flag_recusa = 1

qtd_recusas,tpv_recusado,media,mediana
143,6538.26,45.72,34.98


In [0]:
%sql
-- 4.2 — TPV RECUSADO POR ADQUIRENTE (DETALHADO)
-- Mostra o impacto financeiro da recusa por parceiro
-- % do total indica qual adquirente mais contribui para a perda

SELECT
    adquirente,
    COUNT(*) AS qtd_recusas,
    ROUND(SUM(amount), 2) AS tpv_recusado,
    ROUND(AVG(amount), 2) AS media,
    ROUND(APPROX_PERCENTILE(amount, 0.5), 2) AS mediana,
    ROUND(MIN(amount), 2) AS minimo,
    ROUND(MAX(amount), 2) AS maximo,
    ROUND(SUM(amount) * 100.0 / SUM(SUM(amount)) OVER (), 1) AS pct_do_total
FROM vw_transacoes_unificadas
WHERE flag_recusa = 1
GROUP BY adquirente
ORDER BY tpv_recusado DESC

adquirente,qtd_recusas,tpv_recusado,media,mediana,minimo,maximo,pct_do_total
Adquirente A,51,2272.47,44.56,32.5,1.0,271.7,34.8
Adquirente B,48,2105.74,43.87,35.88,0.01,243.87,32.2
Adquirente C,23,1135.83,49.38,39.3,2.0,156.09,17.4
Adquirente D,14,981.62,70.12,47.89,18.24,329.99,15.0
Não identificado,7,42.6,6.09,4.95,4.95,12.9,0.7


In [0]:
%sql
-- 4.3 — APROVADO vs RECUSADO POR ADQUIRENTE (COMPARAÇÃO DE VALOR)
-- Se o ticket médio de recusa é muito diferente do de aprovação,
-- indica que a recusa não é aleatória — tem padrão de valor

SELECT
    adquirente,
    ROUND(AVG(CASE WHEN flag_sucesso = 1 THEN amount END), 2) AS media_aprovado,
    ROUND(AVG(CASE WHEN flag_recusa = 1 THEN amount END), 2) AS media_recusado,
    ROUND(ABS(AVG(CASE WHEN flag_sucesso = 1 THEN amount END)
         - AVG(CASE WHEN flag_recusa = 1 THEN amount END)), 2) AS diferenca_medias
FROM vw_transacoes_unificadas
GROUP BY adquirente
ORDER BY adquirente

adquirente,media_aprovado,media_recusado,diferenca_medias
Adquirente A,51.07,44.56,6.51
Adquirente B,49.02,43.87,5.15
Adquirente C,46.55,49.38,2.84
Adquirente D,41.82,70.12,28.3
Não identificado,5.59,6.09,0.49


In [0]:
%sql
-- 4.4 — TPV RECUPERÁVEL vs NÃO RECUPERÁVEL
-- Código 51 (saldo insuficiente): RECUPERÁVEL — cliente pode trocar método
-- Código 91 (emissor fora do ar): RECUPERÁVEL — retry/fallback resolve
-- Código 5 (do not honor): AMBÍGUO — pode ser fraude, cartão bloqueado, etc.

SELECT
    recuperabilidade,
    COUNT(*) AS qtd,
    ROUND(SUM(amount), 2) AS tpv_recusado,
    ROUND(SUM(amount) * 100.0 / SUM(SUM(amount)) OVER (), 1) AS pct_do_total
FROM vw_transacoes_unificadas
WHERE flag_recusa = 1
GROUP BY recuperabilidade
ORDER BY tpv_recusado DESC

recuperabilidade,qtd,tpv_recusado,pct_do_total
Recuperável,100,4863.05,74.4
Ambíguo/Não recuperável,43,1675.21,25.6


In [0]:
%sql
-- 4.5 — RECUPERÁVEL vs NÃO RECUPERÁVEL POR ADQUIRENTE
-- Mostra qual adquirente tem mais TPV com chance de recuperação
-- pct_do_total: participação de cada combinação no TPV recusado geral

SELECT
    adquirente,
    recuperabilidade,
    COUNT(*) AS qtd,
    ROUND(SUM(amount), 2) AS tpv,
    ROUND(SUM(amount) * 100.0 / SUM(SUM(amount)) OVER (), 1) AS pct_do_total
FROM vw_transacoes_unificadas
WHERE flag_recusa = 1
GROUP BY adquirente, recuperabilidade
ORDER BY adquirente, recuperabilidade

adquirente,recuperabilidade,qtd,tpv,pct_do_total
Adquirente A,Recuperável,51,2272.47,34.8
Adquirente B,Ambíguo/Não recuperável,36,1480.01,22.6
Adquirente B,Recuperável,12,625.73,9.6
Adquirente C,Ambíguo/Não recuperável,3,175.4,2.7
Adquirente C,Recuperável,20,960.43,14.7
Adquirente D,Recuperável,14,981.62,15.0
Não identificado,Ambíguo/Não recuperável,4,19.8,0.3
Não identificado,Recuperável,3,22.8,0.3


### Conclusões Q4 — TPV recusado e o que representa para o negócio

**TPV recusado total:** R$ 6.538,26 (14,3% do TPV processado de R$ 45.725,02)

**Distribuição por adquirente:**
- Adquirente A: R$ 2.272,47 (34,8% do total recusado) — maior volume absoluto
- Adquirente B: R$ 2.105,74 (32,2%) — segundo maior, mas com perfil de erro diferente
- Adquirente D: R$ 981,62 (15,0%) — menor volume, mas com **maior % de TPV recusado (25,9%)** e **ticket médio de recusa de R$ 70,12** (67% acima do ticket de aprovação de R$ 41,82)

**Recuperável vs não recuperável:**
- **Recuperável (códigos 51 + 91): R$ 4.863,05 (74,4%)** — pode voltar com retry automático ou oferta de método alternativo
- **Não recuperável (código 5): R$ 1.675,21 (25,6%)** — exige investigação com emissor
- Se o TPV recuperável fosse recuperado, a conversão subiria de **85,7% para ~95,7%**

**Por adquirente:**
- A e D: 100% do TPV recusado é recuperável
- C: 84,6% recuperável
- B: apenas **29,7% recuperável** — 70,3% é código 5 (do not honor), o mais difícil de resolver

**O que isso representa para o negócio:**
O TPV recusado não é apenas uma métrica de perda — é um indicador de **oportunidade de recuperação**. A maior parte do que foi recusado (74,4%) tem ação direta por parte do iFood. A diferença entre adquirentes mostra que não existe uma solução única: cada uma precisa de um tipo de ação diferente, conforme o código de erro dominante.

## Fase 5 — Performance dos Parceiros (Q5)

Avaliação multidimensional da performance de cada adquirente, indo além da conversão.
Eixos de análise: perfil de erro, recuperabilidade, normalização (controle de negócio + método),
distribuição por faixa de valor, padrão de aquecimento e governança de dados.

In [0]:
%sql
-- 5.1 — MATRIZ RESUMO DE PERFORMANCE POR ADQUIRENTE
-- Visão multidimensional em uma única tabela:
-- conversão, TPV, % recusado, % recuperável, código dominante e ticket de recusa

SELECT
    adquirente,
    COUNT(*) AS transacoes,
    ROUND(SUM(flag_sucesso) * 100.0 / COUNT(*), 1) AS conv_pct,
    ROUND(SUM(amount), 2) AS tpv_total,
    ROUND(SUM(CASE WHEN flag_recusa = 1 THEN amount ELSE 0 END), 2) AS tpv_recusado,
    ROUND(SUM(CASE WHEN flag_recusa = 1 THEN amount ELSE 0 END) * 100.0 / SUM(amount), 1) AS pct_tpv_recusado,
    ROUND(SUM(CASE WHEN recuperabilidade = 'Recuperável' THEN amount ELSE 0 END)
          * 100.0 / SUM(CASE WHEN flag_recusa = 1 THEN amount ELSE 0 END), 1) AS pct_recuperavel,
    ROUND(AVG(CASE WHEN flag_recusa = 1 THEN amount END), 2) AS ticket_recusado
FROM vw_transacoes_unificadas
WHERE adquirente != 'Não identificado'
GROUP BY adquirente
ORDER BY tpv_recusado DESC

adquirente,transacoes,conv_pct,tpv_total,tpv_recusado,pct_tpv_recusado,pct_recuperavel,ticket_recusado
Adquirente A,400,87.3,20094.53,2272.47,11.3,100.0,44.56
Adquirente B,280,82.9,13479.3,2105.74,15.6,29.7,43.87
Adquirente C,170,86.5,7978.45,1135.83,14.2,84.6,49.38
Adquirente D,81,82.7,3783.49,981.62,25.9,100.0,70.12


In [0]:
%sql
-- 5.2 — PERFIL DE ERRO POR ADQUIRENTE (QUEM É O RESPONSÁVEL)

SELECT
    adquirente,
    return_code,
    COUNT(*) AS qtd_recusas,
    ROUND(SUM(amount), 2) AS tpv_recusado
FROM vw_transacoes_unificadas
WHERE flag_recusa = 1
  AND adquirente != 'Não identificado'
GROUP BY adquirente, return_code
ORDER BY adquirente, return_code

-- Cada adquirente tem um perfil de erro diferente:
-- A: 100% código 51 (saldo do cliente) — não é problema da adquirente
-- B: 75% código 5 (do not honor) — emissor ambíguo
-- C: 87% código 91 (emissor fora do ar) — infraestrutura
-- D: 100% código 91 (emissor fora do ar) — infraestrutura

adquirente,return_code,qtd_recusas,tpv_recusado
Adquirente A,51.0,51,2272.47
Adquirente B,5.0,36,1480.01
Adquirente B,51.0,12,625.73
Adquirente C,5.0,3,175.4
Adquirente C,91.0,20,960.43
Adquirente D,91.0,14,981.62


In [0]:
%sql
-- 5.3 — NORMALIZAÇÃO: A vs B (MESMO NEGÓCIO + MESMO MÉTODO)
-- Para isolar o efeito adquirente do efeito negócio/método,
-- comparamos A e B apenas no mesmo recorte (mesmo negócio + mesmo método)
-- Se B for pior no mesmo recorte, o problema é da adquirente, não do mix

SELECT
    negocio_ifood,
    payment_method,
    adquirente,
    COUNT(*) AS total,
    SUM(flag_sucesso) AS sucessos,
    ROUND(SUM(flag_sucesso) * 100.0 / COUNT(*), 1) AS conv_pct
FROM vw_transacoes_unificadas
WHERE adquirente IN ('Adquirente A', 'Adquirente B')
GROUP BY negocio_ifood, payment_method, adquirente
HAVING COUNT(*) >= 5
ORDER BY negocio_ifood, payment_method, adquirente

negocio_ifood,payment_method,adquirente,total,sucessos,conv_pct
IFOOD_BR_DELIVERY,CREDIT,Adquirente A,208,181,87.0
IFOOD_BR_DELIVERY,CREDIT,Adquirente B,153,125,81.7
IFOOD_BR_DELIVERY,DEBIT,Adquirente A,48,41,85.4
IFOOD_BR_DELIVERY,DEBIT,Adquirente B,42,36,85.7
IFOOD_BR_DELIVERY,DIGITAL_WALLET,Adquirente A,8,6,75.0
IFOOD_BR_DELIVERY,PIX,Adquirente A,103,91,88.3
IFOOD_BR_DELIVERY,PIX,Adquirente B,65,54,83.1
IFOOD_BR_MARKET,CREDIT,Adquirente A,20,18,90.0
IFOOD_BR_MARKET,CREDIT,Adquirente B,7,6,85.7
IFOOD_BR_MARKET,DEBIT,Adquirente A,6,6,100.0


In [0]:
%sql
-- 5.4 — ADQUIRENTE D: CONVERSÃO POR FAIXA DE VALOR
-- O D recusa transações de valor mais alto (ticket recusado R$ 70 vs aprovado R$ 42)
-- Esta tabela mostra a evidência: conversão cai conforme o valor aumenta

SELECT
    CASE
        WHEN amount < 20   THEN '0-20'
        WHEN amount < 50   THEN '20-50'
        WHEN amount < 100  THEN '50-100'
        WHEN amount < 200  THEN '100-200'
        ELSE '200+'
    END AS faixa_valor,
    COUNT(*) AS total,
    SUM(flag_sucesso) AS sucessos,
    ROUND(SUM(flag_sucesso) * 100.0 / COUNT(*), 1) AS conv_pct,
    ROUND(SUM(CASE WHEN flag_recusa = 1 THEN amount ELSE 0 END), 2) AS tpv_recusado
FROM vw_transacoes_unificadas
WHERE adquirente = 'Adquirente D'
GROUP BY CASE
        WHEN amount < 20   THEN '0-20'
        WHEN amount < 50   THEN '20-50'
        WHEN amount < 100  THEN '50-100'
        WHEN amount < 200  THEN '100-200'
        ELSE '200+'
    END
ORDER BY faixa_valor

faixa_valor,total,sucessos,conv_pct,tpv_recusado
0-20,17,16,94.1,18.24
100-200,3,2,66.7,128.99
20-50,36,28,77.8,301.44
200+,1,0,0.0,329.99
50-100,24,21,87.5,202.96


In [0]:
%sql
-- 5.5 — PADRÃO DE AQUECIMENTO (1º vs 2º DIA)

WITH dias_adquirente AS (
    SELECT
        adquirente,
        transaction_date,
        COUNT(*) AS volume,
        SUM(flag_sucesso) AS sucessos,
        ROW_NUMBER() OVER (PARTITION BY adquirente ORDER BY transaction_date) AS dia_ordem
    FROM vw_transacoes_unificadas
    WHERE adquirente != 'Não identificado'
    GROUP BY adquirente, transaction_date
)
SELECT
    adquirente,
    dia_ordem,
    transaction_date,
    volume,
    sucessos,
    ROUND(sucessos * 100.0 / volume, 1) AS conv_pct
FROM dias_adquirente
WHERE dia_ordem <= 2
ORDER BY adquirente, dia_ordem

-- Cada adquirente tem pior conversão no dia 1 de operação e melhora no dia 2
-- O salto de volume do dia 1 para o dia 2 é desproporcional (ex: C passa de 18 para 152)
-- Hipótese: o 1º dia reflete o início de operação da adquirente (ainda não estabilizou)

adquirente,dia_ordem,transaction_date,volume,sucessos,conv_pct
Adquirente A,1,2024-01-22,46,41,89.1
Adquirente A,2,2024-01-23,81,75,92.6
Adquirente B,1,2024-01-25,54,41,75.9
Adquirente B,2,2024-01-26,226,191,84.5
Adquirente C,1,2024-01-26,18,13,72.2
Adquirente C,2,2024-01-29,152,134,88.2
Adquirente D,1,2024-01-29,81,67,82.7


In [0]:
%sql
-- 5.6 — GOVERNANÇA: IFOOD_BR_CLUB SEM ADQUIRENTE


SELECT
    negocio_ifood,
    adquirente,
    COUNT(*) AS transacoes,
    ROUND(SUM(amount), 2) AS tpv
FROM vw_transacoes_unificadas
GROUP BY negocio_ifood, adquirente
ORDER BY negocio_ifood, adquirente

-- Todas as 69 transações de CLUB não têm adquirente mapeado
-- Isso impede avaliar a performance dessa linha de negócio

negocio_ifood,adquirente,transacoes,tpv
IFOOD_BR_CLUB,Não identificado,69,389.25
IFOOD_BR_DELIVERY,Adquirente A,367,16014.74
IFOOD_BR_DELIVERY,Adquirente B,263,11721.41
IFOOD_BR_DELIVERY,Adquirente C,156,6871.51
IFOOD_BR_DELIVERY,Adquirente D,74,3291.03
IFOOD_BR_MARKET,Adquirente A,33,4079.79
IFOOD_BR_MARKET,Adquirente B,17,1757.89
IFOOD_BR_MARKET,Adquirente C,14,1106.94
IFOOD_BR_MARKET,Adquirente D,7,492.46


### Conclusões Q5 — Performance dos parceiros

**Matriz multidimensional:**
Cada adquirente tem um perfil de erro e de perda completamente diferente:

| Adquirente | Conv | TPV Recusado | %Rec | Recup% | Cod dominante | Ticket médio recusado |
|---|---|---|---|---|---|---|
| A | 87,2% | R$ 2.272 | 11,3% | 100% | 51 (saldo) | R$ 44,56 |
| B | 82,9% | R$ 2.106 | 15,6% | 29,7% | 5 (do not honor) | R$ 43,87 |
| C | 86,5% | R$ 1.136 | 14,2% | 84,6% | 91 (emissor fora) | R$ 49,38 |
| D | 82,7% | R$ 982 | 25,9% | 100% | 91 (emissor fora) | R$ 70,12 |

**Perfil de erro por adquirente:**
- **A:** 100% das recusas é código 51 (saldo do cliente) — não é problema da adquirente
- **B:** 75% é código 5 (do not honor) — recusa ambígua do emissor, mais difícil de resolver
- **C:** 87% é código 91 (emissor fora do ar) — infraestrutura, retry pode resolver
- **D:** 100% é código 91 (emissor fora do ar) — infraestrutura, e ainda recusa valor alto

**Normalização A vs B (controle de negócio + método):**
- No mesmo recorte (Delivery + CREDIT): A tem 87,0% vs B tem 81,7% (diff 5,3pp)
- No mesmo recorte (Delivery + PIX): A tem 88,3% vs B tem 83,1% (diff 5,3pp)
- No débito, A e B são iguais (diff -0,3pp)
- Conclusão: B é genuinamente pior em crédito e PIX — não é efeito de mix

**Adquirente D — recusa de valor alto:**
- Conversão por faixa: 94,1% (R$ 0-20) → 77,8% (R$ 20-50) → 66,7% (R$ 100-200) → 0% (R$ 200+)
- Ticket médio de recusa R$ 70,12 vs R$ 41,82 de aprovação (diferença de R$ 28,30)
- Hipótese: timeout configurado muito baixo para transações de valor alto que exigem mais validações

**Padrão de aquecimento:**
- A: dia 1 conv 89,1% (46 trans) → dia 2 conv 92,6% (81 trans) | volume 1,8x
- B: dia 1 conv 75,9% (54 trans) → dia 2 conv 84,5% (226 trans) | volume 4,2x
- C: dia 1 conv 72,2% (18 trans) → dia 2 conv 88,2% (152 trans) | volume 8,4x
- D: apenas 1 dia — não é possível avaliar
- Todas melhoram do 1º para o 2º dia — o 1º dia reflete instabilidade de início de operação
- Quanto maior o salto de volume, maior a recuperação de conversão — o salto acelera a estabilização, não agrava
- Limitação: com apenas 2 dias por adquirente (exceto A), não é possível confirmar se o padrão se sustenta ao longo do tempo

**Governança:**
- Todas as 69 transações de IFOOD_BR_CLUB não têm adquirente mapeado (R$ 389,25 de TPV)
- Sem rastreabilidade, não é possível avaliar a performance dessa linha de negócio
- Problema 100% interno — não depende de parceiro

## Fase 6 — Oportunidades de Melhoria (Q6)

Identificação de oportunidades, ações internas e com parceiros, priorização por impacto e esforço,
e proposta de acompanhamento via scorecard. As consultas abaixo geram a evidência que sustenta
cada recomendação.

In [0]:
%sql
-- 6.1 — ADQUIRENTE B: CONCENTRAÇÃO DE CÓDIGO 5 POR EMISSOR
-- 33% das recusas de código 5 do B estão concentradas nos emissores 1 e 341
-- Isso abre um eixo de investigação dirigida: pode ser incompatibilidade
-- entre o B e esses emissores específicos

SELECT
    card_issuer_code,
    COUNT(*) AS total_trans,
    SUM(CASE WHEN return_code = 5 THEN 1 ELSE 0 END) AS qtd_cod5,
    ROUND(SUM(CASE WHEN return_code = 5 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_cod5,
    ROUND(SUM(CASE WHEN return_code = 5 THEN amount ELSE 0 END), 2) AS tpv_cod5
FROM vw_transacoes_unificadas
WHERE adquirente = 'Adquirente B'
GROUP BY card_issuer_code
HAVING COUNT(*) >= 3
ORDER BY tpv_cod5 DESC

card_issuer_code,total_trans,qtd_cod5,pct_cod5,tpv_cod5
1.0,34,6,17.6,386.35
null,95,12,12.6,305.33
341.0,33,6,18.2,224.32
33.0,16,2,12.5,184.18
260.0,32,4,12.5,121.44
102.0,4,2,50.0,67.94
237.0,16,1,6.3,57.85
801.0,5,1,20.0,49.99
77.0,3,1,33.3,46.73
735.0,3,0,0.0,0.0


In [0]:
%sql
-- 6.1 — ADQUIRENTE B: QUAIS EMISSORES MAIS DISPARAM O CÓDIGO 5


SELECT
    card_issuer_code,
    COUNT(*) AS total_trans_com_b,
    SUM(CASE WHEN return_code = 5 THEN 1 ELSE 0 END) AS qtd_cod5,
    ROUND(SUM(CASE WHEN return_code = 5 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_cod5,
    ROUND(SUM(CASE WHEN return_code = 5 THEN amount ELSE 0 END), 2) AS tpv_cod5
FROM vw_transacoes_unificadas
WHERE adquirente = 'Adquirente B'
  AND card_issuer_code IS NOT NULL
GROUP BY card_issuer_code
HAVING SUM(CASE WHEN return_code = 5 THEN 1 ELSE 0 END) > 0
ORDER BY qtd_cod5 DESC

-- O código 5 (do not honor) é a principal recusa do B (75% das recusas)
-- Esta tabela mostra que a recusa não é distribuída igualmente entre emissores está concentrada em poucos, o que permite uma investigação dirigida

card_issuer_code,total_trans_com_b,qtd_cod5,pct_cod5,tpv_cod5
341.0,33,6,18.2,224.32
1.0,34,6,17.6,386.35
260.0,32,4,12.5,121.44
102.0,4,2,50.0,67.94
33.0,16,2,12.5,184.18
237.0,16,1,6.3,57.85
801.0,5,1,20.0,49.99
422.0,2,1,50.0,35.88
77.0,3,1,33.3,46.73


In [0]:
%sql
-- 6.2 — ADQUIRENTE D: DETALHE DAS RECUSAS (VALOR ALTO)
-- Evidência de que o D recusa transações de valor mais alto
-- Todas as 14 recusas são código 91 (emissor fora do ar)
-- Hipótese: timeout muito baixo para transações que exigem mais validações

SELECT
    card_brand,
    sum(amount) as total_amount,
    return_code,
    payment_method
    
FROM vw_transacoes_unificadas
WHERE adquirente = 'Adquirente D' AND flag_recusa = 1
GROUP BY ALL
ORDER BY total_amount DESC


card_brand,total_amount,return_code,payment_method
MASTERCARD,536.87,91.0,CREDIT
VISA,325.24000000000007,91.0,CREDIT
PIX,51.0,91.0,PIX
AMEX,47.89,91.0,CREDIT
MASTERCARD_MAESTRO,20.62,91.0,DEBIT


In [0]:
%sql
-- 6.3 — ADQUIRENTE A: RECUSAS POR MÉTODO (PARA OFERTA DE ALTERNATIVA)

SELECT
    adquirente,
    return_code,
    payment_method,
    COUNT(*) AS qtd_recusas,
    ROUND(SUM(amount), 2) AS tpv_recusado
FROM vw_transacoes_unificadas
WHERE adquirente = 'Adquirente A' AND flag_recusa = 1
GROUP BY ALL
ORDER BY tpv_recusado DESC

-- 100% das recusas do A são código 51 (saldo insuficiente do cliente)
-- Ação: oferecer método alternativo (PIX, crédito iFood, parcelamento)
-- Esta tabela mostra onde a oferta de alternativa teria mais impacto

adquirente,return_code,payment_method,qtd_recusas,tpv_recusado
Adquirente A,51.0,CREDIT,29,1407.04
Adquirente A,51.0,PIX,13,627.04
Adquirente A,51.0,DEBIT,7,226.45
Adquirente A,51.0,DIGITAL_WALLET,2,11.94


In [0]:
%sql
-- 6.4 — IMPACTO POTENCIAL DE RECUPERAÇÃO POR ADQUIRENTE


SELECT
    adquirente,
    COUNT(*) AS qtd_recusas,
    ROUND(SUM(amount), 2) AS tpv_recusado_total,
    ROUND(SUM(CASE WHEN recuperabilidade = 'Recuperável' THEN amount ELSE 0 END), 2) AS tpv_recuperavel,
    ROUND(SUM(CASE WHEN recuperabilidade = 'Recuperável' THEN amount ELSE 0 END)
          * 100.0 / SUM(amount), 1) AS pct_recuperavel
FROM vw_transacoes_unificadas
WHERE flag_recusa = 1
  AND adquirente != 'Não identificado'
GROUP BY adquirente
ORDER BY tpv_recuperavel DESC

-- Resume o TPV recuperável por adquirente para sustentar a priorização
-- Ordenado pelo maior TPV recuperável — as primeiras linhas são onde a ação de recuperação tem maior ganho financeiro potencial

adquirente,qtd_recusas,tpv_recusado_total,tpv_recuperavel,pct_recuperavel
Adquirente A,51,2272.47,2272.47,100.0
Adquirente D,14,981.62,981.62,100.0
Adquirente C,23,1135.83,960.43,84.6
Adquirente B,48,2105.74,625.73,29.7


### Conclusões Q6 — Oportunidades de melhoria e priorização

**Oportunidades identificadas:**

| # | Oportunidade | Evidência | Tipo de ação |
|---|---|---|---|
| 1 | Retry automático para códigos 91 (C e D) | 100% das recusas de C e D são recuperáveis por retry | Interna (tech) |
| 2 | Oferta de método alternativo para código 51 (A) | 100% das recusas do A são saldo insuficiente; cliente pode trocar método | Interna (produto) |
| 3 | Roteamento alternativo para código 5 (B) | 36 recusas de código 5 do B poderiam ser re-roteadas para outra adquirente | Interna (tech) |
| 4 | Diagnóstico estruturado com Adquirente D | D recusa transações de valor alto (ticket R$ 70 vs R$ 42); hipótese de timeout | Externa (parceiro) |
| 5 | Investigação com emissores 1 e 341 (via B) | 33% das recusas de código 5 do B estão concentradas nesses emissores | Externa (emissor) |
| 6 | Mapeamento de adquirente para IFOOD_BR_CLUB | 69 transações sem rastreabilidade de adquirente | Interna (dados) |

**Matriz de priorização:**

| Prio | Ação | Impacto (TPV) | Esforço | Quem |
|---|---|---|---|---|
| P1 | Retry automático cód 91 (C e D) | R$ 1.955 | Baixo | Interno (tech) |
| P2 | Oferta de método alternativo cód 51 (A) | R$ 2.272 | Médio | Interno (produto) |
| P3 | Roteamento alternativo cód 5 (B) | ~R$ 1.480 | Médio | Interno (tech) |
| P4 | Diagnóstico estruturado com D | R$ 982 | Médio | Externo (parceiro) |
| P5 | Investigação emissores 1 e 341 (via B) | ~R$ 560 | Alto | Externo (emissor) |
| P6 | Mapear adquirente IFOOD_BR_CLUB | R$ 389 | Baixo | Interno (dados) |

**Justificativa da ordem:**
- P1 e P2 primeiro: maior impacto financeiro recuperável + controle total do iFood
- P3: também é interno, mas tem incerteza (retry pode não funcionar para código 5)
- P4 e P5: precisam de parceiro externo, dependem de agenda de terceiros
- P6: baixo impacto financeiro, mas 100% interno e resolve problema de governança

**Scorecard de acompanhamento:**

Indicadores diários:
- Conversão por adquirente
- TPV recusado por adquirente e por código de erro
- % TPV recuperável vs não recuperável

Indicadores semanais:
- Distribuição de códigos de erro por adquirente
- Ticket médio de recusa vs aprovação por adquirente

Gatilhos de alerta:
- Conversão de qualquer adquirente cai mais de 5pp de um dia para o outro
- TPV recusado de qualquer adquirente excede 15% do TPV total processado por ela
- Aparecimento de código de erro novo no histórico